In [1]:
import anndata as ad

st_data = ad.read_h5ad('/work/magroup/skrieger/Syndri/data/xenium_noise_experiment/GSM9509158_breast_breast_B15_B15_rctd_annotated_dropout_1pct.h5ad')

In [ ]:
st_data

In [ ]:
st_data = st_data[st_data.obs['harmony_cell_type'].str.contains('T')]

# st_data = st_data[st_data.obs.groupby('cell_type').sample(frac=0.1).index]

In [ ]:
# st_data.layers['split_purified_counts_X']
# st_data.X

In [ ]:
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
split_deltas =  (st_data.layers['split_purified_counts_trial_0'] - st_data.layers['trial_0'] ).toarray().ravel() #/ (st_data.layers['trial_0'].toarray().ravel()+np.finfo(float).eps)

syndri_probs = st_data.layers['admix_probs_trial_0'].toarray().ravel()

fig,ax = plt.subplots(figsize=(7,5),dpi=250)
sns.scatterplot(x=syndri_probs,y=split_deltas,s=1,ax=ax,alpha=0.05)
ax.set_xlabel('Syndri Probabilities')
ax.set_ylabel('SPLIT - Raw Counts')
ax.set_title('Xenium Breast Cancer B1_3: Syndri Probs vs SPLIT Deltas')
# sns.kdeplot(x=syndri_probs,y=split_deltas, levels=5, color="blue", linewidths=1.5,ax=ax)


In [ ]:
import numpy as np

prob_matrix = st_data.layers["admix_probs_trial_0"].toarray()
raw_matrix = st_data.layers["trial_0"].toarray()

expressed_mask = raw_matrix > 0
expressed_probs = prob_matrix[expressed_mask]

print("Fraction of matrix expressed:", expressed_mask.mean())
print(
    "Fraction of expressed entries with probability exactly zero:",
    np.mean(expressed_probs == 0),
)

print("Expressed-entry mean:", expressed_probs.mean())
print("Expressed-entry std:", expressed_probs.std())
print(
    "Expressed-entry quantiles:",
    np.quantile(
        expressed_probs,
        [0, 0.01, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99, 1],
    ),
)

print(
    "Nonzero probabilities assigned to unexpressed genes:",
    np.mean(prob_matrix[~expressed_mask] > 0),
)

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4), dpi=200)

ax.hist(
    expressed_probs,
    bins=100,
    density=True,
)

ax.set_xlabel("Syndri admixture probability")
ax.set_ylabel("Density")
ax.set_title("Syndri probabilities for expressed genes")

plt.tight_layout()
plt.show()

In [ ]:
masked_probs = np.where(
    expressed_mask,
    prob_matrix,
    np.nan,
)

mean_by_cell = np.nanmean(masked_probs, axis=1)
mean_by_gene = np.nanmean(masked_probs, axis=0)

print("Mean expressed-entry probability:", np.nanmean(masked_probs))
print("Cell-mean SD:", np.nanstd(mean_by_cell))
print("Gene-mean SD:", np.nanstd(mean_by_gene))
y = prob_matrix[expressed_mask]
overall_mean = y.mean()

# Gene-only prediction: assign each observation its gene's average probability
gene_prediction_matrix = np.broadcast_to(
    mean_by_gene[None, :],
    prob_matrix.shape,
)

gene_predictions = gene_prediction_matrix[expressed_mask]

gene_only_r2 = 1 - (
    np.sum((y - gene_predictions) ** 2)
    / np.sum((y - overall_mean) ** 2)
)

# Cell-only prediction
cell_prediction_matrix = np.broadcast_to(
    mean_by_cell[:, None],
    prob_matrix.shape,
)

cell_predictions = cell_prediction_matrix[expressed_mask]

cell_only_r2 = 1 - (
    np.sum((y - cell_predictions) ** 2)
    / np.sum((y - overall_mean) ** 2)
)

print(f"Variation explained by gene identity: {gene_only_r2:.3f}")
print(f"Variation explained by cell identity: {cell_only_r2:.3f}")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

k = 6
cell_type_col = "rctd_first_type"

coords = st_data.obs[["x_centroid","y_centroid"]].values
cell_types = st_data.obs[cell_type_col].astype(str)

neighbor_model = NearestNeighbors(
    n_neighbors=k + 1,
    metric="euclidean",
)

neighbor_indices = (
    neighbor_model
    .fit(coords)
    .kneighbors(return_distance=False)
)

# Remove each cell itself
neighbor_indices = neighbor_indices[:, 1:]

neighbor_types = cell_types.to_numpy()[neighbor_indices]

neighbor_categories = sorted(cell_types.dropna().unique())

neighbor_composition = pd.DataFrame(
    {
        f"neighbor_prop__{category}":
        (neighbor_types == category).mean(axis=1)
        for category in neighbor_categories
    },
    index=st_data.obs_names,
)

neighbor_composition.head()

In [ ]:
import scipy.sparse as sp

raw_layer = st_data.layers["trial_0"]
prob_layer = st_data.layers["admix_probs_trial_0"]

raw_coo = sp.coo_matrix(raw_layer)
prob_csr = sp.csr_matrix(prob_layer)

# One row for every expressed cell-gene observation
cell_indices = raw_coo.row
gene_indices = raw_coo.col
raw_counts = raw_coo.data

syndri_scores = np.asarray(
    prob_csr[cell_indices, gene_indices]
).ravel()

evaluation_df = pd.DataFrame({
    "cell_index": cell_indices,
    "gene_index": gene_indices,
    "cell_id": st_data.obs_names[cell_indices],
    "gene": st_data.var_names[gene_indices],
    "raw_count": raw_counts,
    "syndri_score": syndri_scores,
})

evaluation_df["central_cell_type"] = (
    st_data.obs[cell_type_col]
    .astype(str)
    .to_numpy()[cell_indices]
)

if "sample_replicate" in st_data.obs:
    evaluation_df["sample_replicate"] = (
        st_data.obs["sample_replicate"]
        .astype(str)
        .to_numpy()[cell_indices]
    )
else:
    evaluation_df["sample_replicate"] = "single_sample"

# Logit-transform the scores
epsilon = 1e-5

p = evaluation_df["syndri_score"].clip(
    epsilon,
    1 - epsilon,
)

evaluation_df["syndri_logit"] = np.log(
    p / (1 - p)
)

# Attach each cell's neighborhood composition
evaluation_df = evaluation_df.join(
    neighbor_composition,
    on="cell_id",
)

evaluation_df.head()

In [ ]:
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Optional subsampling for computational tractability
max_rows = 1_000_000

if len(evaluation_df) > max_rows:
    model_df = evaluation_df.sample(
        max_rows,
        random_state=42,
    ).copy()
else:
    model_df = evaluation_df.copy()

neighbor_columns = [
    column
    for column in model_df.columns
    if column.startswith("neighbor_prop__")
]

baseline_categorical = [
    "gene",
    "central_cell_type",
    "sample_replicate",
]

baseline_numeric = [
    "raw_count",
]

full_numeric = baseline_numeric + neighbor_columns

baseline_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=20,
            ),
            baseline_categorical,
        ),
        (
            "numeric",
            StandardScaler(),
            baseline_numeric,
        ),
    ],
)

full_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=20,
            ),
            baseline_categorical,
        ),
        (
            "numeric",
            StandardScaler(),
            full_numeric,
        ),
    ],
)

baseline_model = make_pipeline(
    baseline_preprocessor,
    Ridge(alpha=10.0),
)

full_model = make_pipeline(
    full_preprocessor,
    Ridge(alpha=10.0),
)

X_baseline = model_df[
    baseline_categorical + baseline_numeric
]

X_full = model_df[
    baseline_categorical + full_numeric
]

y = model_df["syndri_logit"].to_numpy()

# Keep all rows from the same cell in the same fold
groups = model_df["cell_id"].to_numpy()

cv = GroupKFold(n_splits=5)

baseline_r2 = []
full_r2 = []

for train_idx, test_idx in cv.split(
    model_df,
    y,
    groups=groups,
):
    baseline_model.fit(
        X_baseline.iloc[train_idx],
        y[train_idx],
    )

    baseline_prediction = baseline_model.predict(
        X_baseline.iloc[test_idx]
    )

    full_model.fit(
        X_full.iloc[train_idx],
        y[train_idx],
    )

    full_prediction = full_model.predict(
        X_full.iloc[test_idx]
    )

    baseline_r2.append(
        r2_score(
            y[test_idx],
            baseline_prediction,
        )
    )

    full_r2.append(
        r2_score(
            y[test_idx],
            full_prediction,
        )
    )

baseline_mean = np.mean(baseline_r2)
full_mean = np.mean(full_r2)
neighbor_delta = full_mean - baseline_mean

print(f"Baseline cross-validated R²: {baseline_mean:.3f}")
print(f"Full cross-validated R²:     {full_mean:.3f}")
print(f"Additional neighbor R²:      {neighbor_delta:.3f}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Extract values
# ------------------------------------------------------------
raw_counts = st_data.layers["trial_0"].toarray().ravel()

split_counts = (
    st_data.layers["split_purified_counts_trial_0"]
    .toarray()
    .ravel()
)

syndri_probs = (
    st_data.layers["admix_probs_trial_0"]
    .toarray()
    .ravel()
)

# Positive values are easier to interpret than SPLIT - raw
counts_removed = raw_counts - split_counts

# Remove invalid observations
valid = (
    np.isfinite(syndri_probs)
    & np.isfinite(counts_removed)
    & (counts_removed >= 0)
)

# Optional but usually recommended:
# compare only gene-cell observations with at least one input count
valid &= raw_counts > 0

comparison_df = pd.DataFrame({
    "syndri_probability": syndri_probs[valid],
    "split_counts_removed": counts_removed[valid],
})


# ------------------------------------------------------------
# Bin Syndri probabilities
# ------------------------------------------------------------
comparison_df = pd.DataFrame({
    "syndri_probability": syndri_probs[valid],
    "raw_counts": raw_counts[valid],
    "split_counts": split_counts[valid],
    "split_counts_removed": counts_removed[valid],
})

comparison_df["fraction_removed"] = (
    comparison_df["split_counts_removed"]
    / comparison_df["raw_counts"]
).clip(0, 1)

# Whether SPLIT removed the entire observed count
comparison_df["completely_removed"] = np.isclose(
    comparison_df["split_counts"],
    0,
)

probability_edges = np.linspace(0, 1, 11)

comparison_df["probability_bin"] = pd.cut(
    comparison_df["syndri_probability"],
    bins=probability_edges,
    include_lowest=True,
)

bin_summary = (
    comparison_df
    .groupby("probability_bin", observed=True)
    .agg(
        mean_probability=("syndri_probability", "mean"),
        mean_fraction_removed=("fraction_removed", "mean"),
        median_fraction_removed=("fraction_removed", "median"),
        q25_fraction_removed=(
            "fraction_removed",
            lambda x: x.quantile(0.25),
        ),
        q75_fraction_removed=(
            "fraction_removed",
            lambda x: x.quantile(0.75),
        ),
        fraction_completely_removed=("completely_removed", "mean"),
        mean_counts_removed=("split_counts_removed", "mean"),
        n_observations=("split_counts_removed", "size"),
    )
    .reset_index()
)

bin_summary["probability_midpoint"] = (
    bin_summary["probability_bin"]
    .map(lambda interval: interval.mid)
    .astype(float)
)


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5),
    dpi=250,
)

# Panel 1: density of individual gene-cell observations
hexbin = axes[0].hexbin(
    comparison_df["syndri_probability"],
    comparison_df["split_counts_removed"],
    gridsize=(60, 40),
    bins="log",
    mincnt=1,
    cmap="viridis",
)

colorbar = fig.colorbar(
    hexbin,
    ax=axes[0],
)

colorbar.set_label("log10 number of observations")

axes[0].set_xlabel("Syndri admixture probability")
axes[0].set_ylabel("Counts removed by SPLIT")
axes[0].set_title("Gene-level agreement")

# Optional: prevent rare extreme removals from compressing the plot
upper_limit = comparison_df["split_counts_removed"].quantile(0.995)
axes[0].set_ylim(-0.5, upper_limit)


# Panel 2: conditional probability of SPLIT removing counts
x = bin_summary["probability_midpoint"]
y = bin_summary["mean_fraction_removed"]

axes[1].plot(
    x,
    y,
    marker="o",
    label="Mean fraction removed",
)

axes[1].fill_between(
    x,
    bin_summary["q25_fraction_removed"],
    bin_summary["q75_fraction_removed"],
    alpha=0.2,
    label="Interquartile range",
)

axes[1].set_xlabel("Syndri admixture probability")
axes[1].set_ylabel("Fraction of observed counts removed by SPLIT")
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)
axes[1].set_title("SPLIT removal magnitude by Syndri probability")
axes[1].legend(frameon=False)

fig.suptitle(
    "Xenium Breast Cancer B1_3: Syndri versus SPLIT",
    fontsize=16,
)

fig.tight_layout()
plt.show()

display(bin_summary)